# Waypoint: is this route feasible? (Datathon Task 2B)

Waypoint's planner and its feasibility rules live in one pure-Python package, `waypoint_rules`. The API, the seed checks and this
notebook import the same code, so a number here is the number the dispatcher's screen shows.

This notebook plays the **reference day** from the PRD (section 4c): six vehicles, three districts and the 24 pinned orders. It reads
no competition files. It shows, in order:

1. the booklet's **trip-minutes formula** applied to the released plan v3, and the planned clock for each trip;
2. **`check_trip`**: every hard rule a trip must satisfy, and what a move that breaks them looks like;
3. the **planner's own draft** for the same day next to the hand-made plan.

In [1]:
import sys
from pathlib import Path

# Run from the analytics/ folder. The rules package and the reference day live in backend/.
sys.path.insert(0, str(Path("..") / "backend"))

from tests.rules.conftest import ORDERS, REF, SERVICE, VEHICLE_DAYS, plan_v3
from waypoint_rules import Move, check_trip, draft_plan, hhmm, planned_clock, trip_minutes, validate_move

plan = plan_v3()
print(f"Service day {SERVICE:%a %d %b %Y}: {len(ORDERS)} orders, {len(REF.vehicles)} vehicles, {len(REF.districts)} districts")
print("Plan v3 trips:", ", ".join(f"{v} trip {n}" for v, n in sorted(plan.trips)))
print("Deferred in plan v3:", ", ".join(plan.deferred))

Service day Tue 29 Sep 2026: 24 orders, 6 vehicles, 3 districts
Plan v3 trips: VEH003 trip 1, VEH003 trip 2, VEH011 trip 1, VEH035 trip 1, VEH035 trip 2, VEH039 trip 1
Deferred in plan v3: ORD1020, ORD1009, ORD1017, ORD1006


## 1. Trip minutes

The booklet's formula, counted per order with the return leg left out:

`trip minutes = outbound + inter-stop x (orders - 1) + sum of the handling allowance for each order's (brand, dock type)`

A Fresh (chilled) trip has **270 minutes** of reefer budget and a Style or Tech trip has **480**, over at most two trips a day.

In [2]:
BUDGET = {"Fresh": 270}  # every other brand: 480

print(f"{'trip':<12}{'orders':>7}{'minutes':>9}{'budget':>8}   {'depart':<7}{'back':<7}{'rules broken'}")
for key in sorted(plan.trips):
    trip = plan.trips[key]
    brand = REF.outlets[ORDERS[trip.order_ids[0]].outlet_id].brand.value
    minutes = trip_minutes(trip, ORDERS, REF)
    clock = planned_clock(trip, ORDERS, REF)
    broken = check_trip(trip, ORDERS, REF, VEHICLE_DAYS.get(trip.vehicle_id))
    name = f"{key[0]} trip {key[1]}"
    print(f"{name:<12}{len(trip.order_ids):>7}{minutes:>9}{BUDGET.get(brand, 480):>8}   {hhmm(trip.depart_at):<7}{hhmm(clock.back_at_depot):<7}{len(broken)}")

trip         orders  minutes  budget   depart back   rules broken
VEH003 trip 1      5      132     270   03:30  06:09  0
VEH003 trip 2      4      109     270   06:09  08:22  0
VEH011 trip 1      1       83     480   08:36  10:23  0
VEH035 trip 1      4      125     270   03:30  06:12  0
VEH035 trip 2      3      101     270   06:12  08:30  0
VEH039 trip 1      3       73     270   05:10  06:37  0


The planned clock gives the arrival at every stop. Orders for one outlet share a single stop (one arrival, no hop between them), and a
truck that arrives before a store's window opens waits for it. Here is VEH003's first trip:

In [3]:
trip = plan.trips[("VEH003", 1)]
clock = planned_clock(trip, ORDERS, REF)
print(f"VEH003 trip 1 leaves {hhmm(trip.depart_at)}")
for stop in clock.stops:
    outlet = REF.outlets[ORDERS[stop.order_ids[0]].outlet_id]
    print(f"  {hhmm(stop.arrival)}  {outlet.id}  window {outlet.window_open:%H:%M}-{outlet.window_close:%H:%M}  {', '.join(stop.order_ids)}")
print(f"back at the depot {hhmm(clock.back_at_depot)}")

VEH003 trip 1 leaves 03:30
  03:54  OUT011  window 03:00-08:00  ORD1014
  04:17  OUT006  window 03:00-08:00  ORD1016
  04:41  OUT005  window 04:00-07:45  ORD1011
  05:04  OUT009  window 04:00-07:45  ORD1002
  05:27  OUT012  window 05:30-08:00  ORD1001
back at the depot 06:09


## 2. What breaks a trip

`validate_move` applies a move to a copy of the plan and reports **every** rule it breaks, not just the first. These are the dispatcher's
three refusals in the PRD walkthrough, played on the hand-made plan v3.

In [4]:
moves = {
    "ORD1009 to VEH003 trip 2": Move("ORD1009", ("VEH003", 2)),
    "ORD1002 to VEH011 trip 1": Move("ORD1002", ("VEH011", 1)),
    "ORD1001 to the pool": Move("ORD1001", None),
}
for label, move in moves.items():
    result = validate_move(plan, move, ORDERS, REF, VEHICLE_DAYS)
    print(("allowed" if result.ok else "REFUSED"), "-", label)
    for violation in result.violations:
        print(f"    {violation.rule}: {violation.message}")

REFUSED - ORD1009 to VEH003 trip 2
    R-WINDOW: Arrives 08:06, after OUT014 closes at 08:00
REFUSED - ORD1002 to VEH011 trip 1
    R-TEMP: Needs a reefer: VEH011 is ambient
    R-BRAND: Two brands on one trip: Fresh and Style
REFUSED - ORD1001 to the pool
    R-CONT: OUT012 was deferred yesterday; the continuity guard protects it


## 3. The planner's draft against the hand-made plan

`draft_plan` is a deterministic function of the closed queue, the fleet and the calendar. It groups by brand and district, builds trips on the
best-fitting vehicle, and defers what does not fit as **capacity** (no vehicle can carry it) or **policy** (it would break a rule or a protected
stop), each with a reason. It does not copy plan v3; where they differ, the computed numbers are the ones shown (PRD departure DP-01).

In [5]:
draft = draft_plan(ORDERS, REF, VEHICLE_DAYS, service_date=SERVICE)

print(f"{'trip':<12}{'orders':>7}{'minutes':>9}   rules broken")
for t in draft.trips:
    trip = t.to_trip()
    broken = check_trip(trip, ORDERS, REF, VEHICLE_DAYS.get(t.vehicle_id))
    print(f"{t.vehicle_id + ' trip ' + str(t.trip_no):<12}{len(t.order_ids):>7}{trip_minutes(trip, ORDERS, REF):>9}   {len(broken)}")

print()
print(f"Served {sum(len(t.order_ids) for t in draft.trips)} of {len(ORDERS)} orders; deferred {len(draft.deferrals)}:")
for d in draft.deferrals:
    print(f"  {d.order_id}  {d.type.value:<8} {d.reason_text}")

trip         orders  minutes   rules broken
VEH003 trip 1     10      249   0
VEH035 trip 1      8      222   0
VEH037 trip 1      1       83   0
VEH039 trip 1      3       73   0

Served 22 of 24 orders; deferred 2:
  ORD1017  policy   Fresh minutes over budget: 273 of 270 on VEH003
  ORD1020  capacity van_only and 1,250 kg; the largest Peliyagoda reefer van carries 1,040 kg; whole orders can't split.


In [6]:
same = {k: tuple(v.order_ids) for k, v in plan.trips.items()}
drafted = {(t.vehicle_id, t.trip_no): tuple(t.order_ids) for t in draft.trips}
print("Trips identical to plan v3:", sum(1 for k in same if drafted.get(k) == same[k]), "of", len(same))
print("Every drafted trip breaks no rule:", all(not check_trip(t.to_trip(), ORDERS, REF, VEHICLE_DAYS.get(t.vehicle_id)) for t in draft.trips))
print("Same input, same plan:", draft_plan(ORDERS, REF, VEHICLE_DAYS, service_date=SERVICE).trips == draft.trips)

Trips identical to plan v3: 0 of 6
Every drafted trip breaks no rule: True
Same input, same plan: True


## Reading this

* **Task 2B** asks whether a route is feasible. `check_trip` answers it: a trip is feasible when it returns no violations, and each violation names the rule
  (`R-TEMP`, `R-BRAND`, `R-DISTRICT`, `R-BUDGET-F`, `R-WINDOW`, `R-CONT`, ...) in the words a dispatcher reads.
* The **trip-minutes** column is the booklet's formula, so it can be compared with any competing model's estimate.
* The same package runs behind every screen of the app; the end-to-end tests in `e2e/` check that the refusals a judge sees are these ones.

To run it: `pip install -e "backend[dev]" jupyter`, then open this notebook from the `analytics/` folder.